In [0]:
%pip install kneed

In [0]:
dbutils.library.restartPython()

In [0]:
df = spark.sql("""
SELECT DISTINCT 
order_id,
street_address,
zipcode_int,
latitude,
longitude,
type,
value,
last_processed_time
FROM end_to_end_retail.db_gold.tbg_customer_item_clustering
WHERE zipcode_int IS NOT NULL
""")

In [0]:
# %sql
# SELECT
#   order_id,
#   street_address,
#   zipcode_int,  
#   latitude,
#   longitude,
#   type,
#   value,
#   last_processed_time,
#   ai_query(
#     "databricks-meta-llama-3-3-70b-instruct",
#     CONCAT("Give me the latitude of ", street_address, "make sure the output in only the latitude"))as latitude,
#   ai_query(
#     "databricks-meta-llama-3-3-70b-instruct",
#     CONCAT("Give me the longitude of ", street_address, "make sure the output in only the longitude"))as longitude
# FROM end_to_end_retail.db_gold.tbg_customer_item_clustering
# LIMIT 5


In [0]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

- Priority zipcode table management

In [0]:
# from delta.tables import DeltaTable

# # --- STEP A: Identify and Process the "Vital Few" ---
# # Only look at 'NEW' data or data that hasn't been processed in 7 days
# target_zips_df = spark.table("sales_orders") \
#     .filter("(status = 'NEW') OR (last_updated < current_date() - 7)") \
#     .groupBy("zipcode") \
#     .agg(F.count("*").alias("priority")) \
#     .orderBy(F.desc("priority")) \
#     .limit(100)

# top_zips = [r.zipcode for r in target_zips_df.collect()]

# # Filter original data for these zips
# df_to_process = spark.table("sales_orders").filter(F.col("zipcode").isin(top_zips))

# # Run the Hybrid DBSCAN Function (defined in previous response)
# # Let's assume 'processed_pdf' is the result of that function
# processed_df = spark.createDataFrame(processed_pdf) \
#     .withColumn("status", F.lit("PROCESSED")) \
#     .withColumn("last_updated", F.current_timestamp())

# # --- STEP B: Atomic Update using Delta Merge ---
# # This ensures that while we cluster, no data is lost and assignments are saved.
# master_table = DeltaTable.forName(spark, "sales_orders")

# master_table.alias("target").merge(
#     processed_df.alias("updates"),
#     "target.order_id = updates.order_id"
# ).whenMatchedUpdate(set = {
#     "status": "updates.status",
#     "cluster_id": "updates.cluster_id",
#     "assignment": "updates.assignment",
#     "last_updated": "updates.last_updated"
# }).execute()


#### 1. Initialize Delta Table with Tracking Columns

In [0]:
df = spark.sql("""
SELECT 
order_id,
street_address,
zipcode_int,
latitude,
longitude,
type,
value,
last_processed_time
FROM end_to_end_retail.db_gold.tbg_customer_item_clustering
WHERE zipcode_int IS NOT NULL
""")

##### a.Batch 1

In [0]:
%sql
CREATE OR REPLACE TABLE end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders 
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true)
AS
SELECT 
  order_id,
  street_address,
  CAST(zipcode_int AS STRING) AS zipcode,
  latitude,
  longitude,
  type,
  value,
  -- NULL AS status,
  -- NULL AS cluster_id,
  -- NULL AS assignment,
  last_processed_time AS last_updated
FROM end_to_end_retail.db_gold.tbg_customer_item_clustering
WHERE zipcode_int IS NOT NULL
LIMIT 3000

#### 2. The Dynamic Priority Agent Script
**The Pareto Metric**: The script identifies Zip Codes where the "Sales Volume" and "Maintenance Tickets" are highest. By using a weighted sum, you dynamically prioritize revenue-generating areas over low-activity zones.

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances, haversine_distances
from sklearn.cluster import DBSCAN
from kneed import KneeLocator
import matplotlib.pyplot as plt

# --- STEP 1: CALCULATE PARETO IMPORTANCE ---
importance_df = spark.table("end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders").groupBy("zipcode").agg(
    F.sum(F.when(F.col("type") == "SALES", F.col("value")).otherwise(0)).alias("sales_val"),
    F.count(F.when(F.col("type") == "MAINTENANCE", 1)).alias("maint_count"),
    F.max("last_updated").alias("last_seen")
).withColumn(
    "importance_score", 
    (F.col("sales_val") * 0.7) + (F.col("maint_count") * 0.3)
)

top_zips = [r.zipcode for r in importance_df.orderBy(F.desc("importance_score")).limit(100).collect()]

top_zips_df = pd.DataFrame(top_zips, columns=["zipcode"])
top_zips_spark_df = spark.createDataFrame(top_zips_df)

In [0]:
sample_df = spark.table("end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders")
joined_df = sample_df.join(top_zips_spark_df, sample_df.zipcode == top_zips_spark_df.zipcode, "inner")

# Remove duplicate 'zipcode' column from top_zips_spark_df
joined_df = joined_df.drop(top_zips_spark_df.zipcode)

display(joined_df)

##### b.KMeans

**The Hybrid Cluster Agent**: It combines the physical proximity (Haversine) with the address string (TF-IDF). This prevents an agent from being assigned to two addresses that are physically close but are actually on different street blocks or have vastly different naming conventions.

In [0]:
# --- STEP 2: HYBRID CLUSTERING AGENT (KMeans Version) ---
from sklearn.cluster import KMeans

def agent_clustering_logic(pdf):
    if len(pdf) < 3: return pdf

    tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3,3))
    dist_text = cosine_distances(tfidf.fit_transform(pdf['street_address']))
    coords = np.radians(pdf[['latitude', 'longitude']].values)
    dist_space = haversine_distances(coords)
    dist_hybrid = (0.9 * dist_space) + (0.1 * dist_text)

    # Convert distance matrix to embedding for KMeans (e.g., MDS or use coordinates directly)
    # Here, use latitude/longitude and add text similarity as a feature
    text_sim = 1 - dist_text.mean(axis=1)
    features = np.hstack([pdf[['latitude', 'longitude']].values, text_sim.reshape(-1,1)])

    kmeans = KMeans(n_clusters=3, random_state=43)
    pdf['cluster_id'] = kmeans.fit_predict(features)
    pdf['assignment'] = pdf.apply(lambda r: f"Sales Team A-Cluster {r.cluster_id}" if r['type'] == 'SALES' else f"Tech Team B-Cluster {r.cluster_id}", axis=1)
    pdf['status'] = 'PROCESSED'
    pdf['last_updated'] = pd.Timestamp.now()
    return pdf

In [0]:
# Convert Spark DataFrame to Pandas for clustering logic
joined_pdf = joined_df.toPandas()
processed_pdf = agent_clustering_logic(joined_pdf)

# Filter out negative clusters (-1) before display and plotting
# filtered_pdf = processed_pdf[processed_pdf['cluster_id']]

final_spark_df = spark.createDataFrame(processed_pdf)

In [0]:
final_spark_df.createOrReplaceTempView("final_spark_df")

In [0]:
%sql
SELECT DISTINCT * FROM final_spark_df 

In [0]:
# --- PLOT CLUSTERS ---
plt.figure(figsize=(8,6))
for cluster in processed_pdf['cluster_id'].unique():
    cluster_data = processed_pdf[processed_pdf['cluster_id'] == cluster]
    plt.scatter(cluster_data['longitude'], cluster_data['latitude'], label=f'Cluster {cluster}', alpha=0.6)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Cluster Visualization')
plt.legend()
plt.show()

In [0]:
final_spark_df.write.format("delta").mode("overwrite").saveAsTable("end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders_top100zip")